# MyAI — Kaggle GPU Training (T4/P100, 30h/week)

**Order (each cell runs top→bottom):** 0. GPU check → 1. clone+deps → 2. stream FineWeb 10M → 3. corpus → 4. 32k tokenizer → 5. pack → 6. 20M smoke → 7. 100M run → 8. eval.
Interrupt/stop anytime — checkpoints in `/kaggle/working` survive; resume with `--resume`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
!git clone https://github.com/aloksingh9667/llm.git myai && cd myai && git log --oneline -3
%cd /kaggle/working/myai
!pip install -q -r requirements/train.txt
!python scripts/check_env.py

In [ ]:
# ~10M-token FineWeb sample (ODC-By-1.0, manifest auto-written). ~10-20 min.
!python scripts/stream_fineweb.py --max-tokens 10000000 --out data/raw/fineweb-10M.txt

In [ ]:
!python scripts/prepare_tokenizer_corpus.py
!python -c "import json; m=json.load(open('data/tokenizer_corpus/manifest.json')); print(m['docs_kept'], 'docs,', m['total_chars'], 'chars')"

In [ ]:
# 32k BPE from scratch on the 10M corpus (the real freeze — CPU seed vocab is replaced here).
!python scripts/train_tokenizer.py --corpus data/tokenizer_corpus/train.txt --vocab-size 32000 --out data/tokenizer_corpus/tokenizer-32k.json

In [ ]:
%%writefile configs/data-pack-fw10m.yaml
version: "1.0.0"
corpus_path: data/tokenizer_corpus/train.txt
tokenizer_path: data/tokenizer_corpus/tokenizer-32k.json
output_dir: data/processed/fw10m-32k
seq_len: 512
val_ratio: 0.01
!python scripts/prepare_sequences.py --config configs/data-pack-fw10m.yaml

In [ ]:
# Smoke: MyAI-20M architecture, 200 steps. Watch tok/s → extrapolate the 100M budget.
!python scripts/train.py --model-config configs/myai-20m.yaml --data-dir data/processed/fw10m-32k --max-steps 200 --batch-size 4 --accum 8 --amp --eval-every 100 --ckpt-every 100 --out-dir /kaggle/working/ckpts-20m --report /kaggle/working/train-20m.json

In [ ]:
# Main run: MyAI-100M. Effective batch = 4*8=32 seqs x512 = 16k tokens/step.
# 5000 steps ≈ 82M tokens. Split across sessions; resume with --resume <ckpt>.
!python scripts/train.py --model-config configs/myai-100m.yaml --data-dir data/processed/fw10m-32k --max-steps 5000 --batch-size 4 --accum 8 --amp --eval-every 250 --ckpt-every 500 --out-dir /kaggle/working/ckpts-100m --report /kaggle/working/train-100m.json

In [ ]:
!python scripts/eval_model.py --checkpoint /kaggle/working/ckpts-100m/final.pt --data-dir data/processed/fw10m-32k --report /kaggle/working/eval-100m.json